In [1]:
# Imports
from dotenv import load_dotenv
from anthropic import Anthropic
from building_with_the_claude_api import add_user_message, add_assistant_message, chat, Effort
from building_with_the_claude_api.prompt_evaluator import PromptEvaluator, generate_prompt_evaluation_report


In [2]:
# Client Initialization and helper functions

load_dotenv()

client = Anthropic()

model = "claude-haiku-4-5"
# model = "claude-sonnet-4-6"


In [3]:
# Create an instance of PromptEvaluator
# Increase `max_concurrent_tasks` for greater concurrency, but beware of rate limit errors!
evaluator = PromptEvaluator(max_concurrent_tasks=1)

In [4]:
dataset = evaluator.generate_dataset(
    client=client,
    model=model,
    # Describe the purpose or goal of the prompt you're trying to test
    task_description="write a compact, concise 1 day meal plan for a single athlete",
    # Describe the different inputs that your prompt requires
    prompt_inputs_spec={
        "height": "Athlete height in cm",
        "weight": "Athlete weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete",
    },
    # Number of test cases to generate (recommend keeping this low if you're getting rate limit errors)
    num_cases=3,
)

Generated 1/3 test cases
Generated 2/3 test cases
Generated 3/3 test cases


In [9]:
# Define and run the prompt you want to evaluate, returning the raw model output
# This function is executed once for each test case
def run_prompt(prompt_inputs):
    prompt = f"""
    Generate a one-day meal plan for an athlete that meets their dietary restrictions.

    - Height: {prompt_inputs["height"]}
    - weight: {prompt_inputs["weight"]}
    - Goal: {prompt_inputs["goal"]}
    - Dietary restrictions:  {prompt_inputs["restrictions"]}

    Use either of the following guidelines or steps to respond.
    Guidelines (variant 1 being specific):
    1. Include accurate daily calorie amount
    2. Show protein, fat, and carb amounts
    3. Specify when to eat each meal
    4. Use only foods that fit restrictions
    5. List all portion sizes in grams
    6. Keep budget-friendly if mentioned

    Follow these steps (variant 2 providing steps):
    1. Calculate daily calories needed
    2. Figure out protein, fat, carb amounts
    3. Plan meal timing around workouts
    4. Choose foods that fit restrictions
    5. Set portion sizes in grams
    6. Adjust for budget if needed
    """

    messages = []
    add_user_message(messages, prompt)
    return chat(messages=messages, client=client, model=model, effort=Effort.LOW)


In [11]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt, client=client, model=model, extra_criteria = """
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions and timing
    """
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 6.333333333333333
